In [ ]:
library(loadeR)
library(visualizeR)
library(sp)

In [ ]:
source("../datasets-interactive-atlas/hatching-functions/hatching-functions.R")
source("../datasets-interactive-atlas/hatching-functions/AR6-WGI-hatching.R")

In [ ]:
di <- dataInventory("auxiliary-material/CMIP6_SAM_historical_JJA_pr_1850-1900_annual.nc")
str(di)

In [ ]:
ref <- loadGridData("auxiliary-material/CMIP6_SAM_historical_JJA_pr_1850-1900_annual.nc", var = "pr")
delta <- loadGridData("auxiliary-material/CMIP6_SAM_ssp585_JJA_pr_2081-2100_delta.nc", var = "pr")

In [ ]:
pl.simple.sam <- AR6.WGI.hatching(delta = delta, method = "simple", 
                                  map.hatching.args = list(density = 2, lwd = 1.2), 
                                  color.theme = "BrBG")

In [ ]:
pl.simple.sam

In [ ]:
rel.delta <- loadGridData("auxiliary-material/CMIP6_SAM_ssp585_JJA_pr_2081-2100_relative_delta.nc", var = "pr")
pl.simple.sam <- AR6.WGI.hatching(delta = delta, relative.delta = rel.delta, method = "simple", 
                                  map.hatching.args = list(density = 2, lwd = 1.2), 
                                  color.theme = "BrBG", set.max = 50, set.min = -50)

In [ ]:
pl.simple.sam

In [ ]:
pl.advanced.sam <- AR6.WGI.hatching(delta = delta, historical.ref = ref, relative.delta = rel.delta, method = "advanced", 
                                  map.hatching.args = list(density = 1, lwd = 0.7), 
                                  color.theme = "BrBG", set.max = 50, set.min = -50)

In [ ]:
pl.advanced.sam

In [ ]:
di <- dataInventory("auxiliary-material/CMIP6_historical_JJA_pr_1995-2014.nc")

In [ ]:
hist <- loadGridData("auxiliary-material/CMIP6_historical_JJA_pr_1995-2014.nc", var = "pr")
scen <- loadGridData("auxiliary-material/CMIP6_ssp585_JJA_pr_2081-2100.nc", var = "pr")

In [ ]:
delta <- gridArithmetics(climatology(scen), climatology(hist), operator = "-")
ensemble.mean <- function(grid) aggregateGrid(grid, aggr.mem = list(FUN = mean, na.rm = T))
delta.ens <- ensemble.mean(delta)
hist.ens <- ensemble.mean(climatology(hist))
# Relative delta
rel.delta <- gridArithmetics(delta.ens, 
                             hist.ens, 
                             100, 
                             operator = c("/", "*"))

In [ ]:
spatialPlot(delta, backdrop.theme = "coastline", main = "delta", 
           set.max = 5, set.min = -5, color.theme = "BrBG")

In [ ]:
spatialPlot(rel.delta, backdrop.theme = "coastline", main = "relative delta", 
           set.max = 60, set.min = -60, color.theme = "BrBG")

In [ ]:
simple <- aggregateGrid(delta, aggr.mem = list(FUN = agreement, th = 80))
spatialPlot(simple)

In [ ]:
simple.hatch <- map.hatching(clim = climatology(simple), threshold = "0.5", angle = "-45",
                             condition = "LT", density = 4,  lwd = 0.6,
                             upscaling.aggr.fun = list(FUN = mean))
plot(simple.hatch[[2]])

In [ ]:
pl.simple <- spatialPlot(rel.delta, 
            color.theme = "BrBG", 
            at = seq(-60, 60, 5), 
            set.max = 60, set.min = -60,
            backdrop.theme = "coastline",
            main = list("Mean delta change", cex = 0.8),
            xlab = list("Period: 2081-2100, Season: JJA", cex = 0.8),
            sp.layout = list(simple.hatch),
            par.settings = list(axis.line = list(col = 'transparent')))
pl.simple

In [ ]:
sign <- loadGridData("auxiliary-material/CMIP6_ssp585_JJA_pr_2081-2100_signal.nc", var = "pr_signal")

In [ ]:
advanced1 <- aggregateGrid(sign, aggr.mem = list(FUN = signal.ens1, th = 66))

In [ ]:
advanced2.aux <- aggregateGrid(sign, aggr.mem = list(FUN = signal.ens2, th = 66))

In [ ]:
spatialPlot(advanced1, main = "no signal = 0")
spatialPlot(advanced2.aux, main = "no signal = 1")

In [ ]:
advanced2.aux <- gridArithmetics(advanced2.aux, simple, operator = "+") 
spatialPlot(advanced2.aux)

In [ ]:
advanced2 <- binaryGrid(advanced2.aux, condition = "GT", threshold = 0)
spatialPlot(advanced2)

In [ ]:
my.hatching <- function(grid, angle) map.hatching(grid, threshold = 0.8, angle = angle,
                                       condition = "LT", density = 4,  lwd = 0.6,
                                       upscaling.aggr.fun = list(FUN = mean))
advanced1.hatch <- my.hatching(climatology(advanced1), "-45")
advanced2.hatch <- my.hatching(climatology(advanced2),  "45")
advanced2.hatch.bis <- my.hatching(climatology(advanced2), "-45")

In [ ]:
plot(advanced1.hatch[[2]])
plot(advanced2.hatch[[2]], add = TRUE)
plot(advanced2.hatch.bis[[2]], add = TRUE)

In [ ]:
pl.advanced <- spatialPlot(rel.delta, 
            color.theme = "BrBG", 
            at = seq(-60, 60, 5), 
            set.max = 60, set.min = -60,
            backdrop.theme = "coastline",
            main = list("Mean delta change", cex = 0.8),
            xlab = list("Period: 2081-2100, Season: JJA", cex = 0.8),
            sp.layout = list(advanced1.hatch, advanced2.hatch, advanced2.hatch.bis),
            par.settings = list(axis.line = list(col = 'transparent')))
pl.advanced

In [ ]:
sessionInfo()